# Human Development Index (HDI) Prediction & Socioeconomic EDA
### **Author:** Senior Machine Learning Engineer & Python Architect
### **Project:** Human Development Index (HDI) Prediction System

This notebook contains the complete, production-grade end-to-end Machine Learning pipeline for predicting the Human Development Index (HDI) of countries using essential socioeconomic indicators:
1. **Life Expectancy at Birth** (Longevity index components)
2. **Expected Years of Schooling** (Knowledge enrollment components)
3. **Mean Years of Schooling** (Knowledge attainment components)
4. **Gross National Income (GNI) Per Capita** in PPP USD (Standard of living index components)

## 1. Setup and Package Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Set style for high-quality graphs
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

## 2. Module 1: Dataset Loading & Descriptive Statistics

In [ ]:
# Load HDI Dataset
csv_path = "../dataset/HDI.csv"
if not os.path.exists(csv_path):
    csv_path = "dataset/HDI.csv" # fallback root directory

df = pd.read_csv(csv_path)

# Display dataset shape
print(f"Dataset Shape: {df.shape} (Rows, Columns)\n")

# Display column names
print("Columns:", list(df.columns), "\n")

# Display info
print("Dataset Information:")
print(df.info(), "\n")

# Descriptive statistics
print("Descriptive Statistics:")
print(df.describe(), "\n")

# Missing values analysis
print("Missing Values:")
print(df.isnull().sum(), "\n")

# Duplicate rows check
print(f"Duplicate Rows: {df.duplicated().sum()}")

## 3. Module 2: Exploratory Data Analysis (EDA)

In [ ]:
# Create plots directory if it does not exist
os.makedirs("../plots", exist_ok=True)
os.makedirs("plots", exist_ok=True)

# 3.1 Distribution of HDI (Target Variable)
plt.figure(figsize=(10, 5))
sns.histplot(df["HDI"], kde=True, color="indigo", bins=15)
plt.title("Distribution of Human Development Index (HDI) Scores", fontsize=14, fontweight="bold")
plt.xlabel("HDI Score")
plt.ylabel("Frequency")
plt.savefig("plots/hdi_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 3.2 Feature Correlations Heatmap
numeric_cols = ["Life_Expectancy", "Expected_Schooling", "Mean_Schooling", "GNI_Per_Capita", "HDI"]
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5, cbar=True)
plt.title("Pearson Correlation Coefficients Matrix", fontsize=14, fontweight="bold")
plt.savefig("plots/correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 3.3 Relationship Scatter Plots (Features vs HDI)
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

features = ["Life_Expectancy", "Expected_Schooling", "Mean_Schooling", "GNI_Per_Capita"]
colors = ["tomato", "royalblue", "forestgreen", "darkorange"]

for idx, feat in enumerate(features):
    if feat == "GNI_Per_Capita":
        # logarithmic scaling for income indices matches UN specifications
        sns.scatterplot(x=df[feat], y=df["HDI"], ax=axes[idx], color=colors[idx], alpha=0.85)
        axes[idx].set_xscale("log")
        axes[idx].set_title("GNI per Capita (Log Scaled) vs Target HDI", fontweight="bold")
    else:
        sns.regplot(x=df[feat], y=df["HDI"], ax=axes[idx], color=colors[idx], scatter_kws={"alpha":0.7})
        axes[idx].set_title(f"{feat.replace('_', ' ')} vs Target HDI", fontweight="bold")
    
plt.suptitle("Socioeconomic Indicators Regression vs Target HDI", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/features_vs_hdi_scatters.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Module 3: Data Preprocessing & Scaling

In [ ]:
# Separate features and target variables
X = df[features]
y = df["HDI"]

# Perform Train Test Split (75% Train, 25% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Training set dimension: {X_train.shape}")
print(f"Testing set dimension: {X_test.shape}")

# Feature Scaling (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("First 3 normalized training samples:\n", X_train_scaled[:3])

## 5. Module 4: Machine Learning Pipelines & Evaluation

In [ ]:
# Initialize four regression models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=4, min_samples_split=5, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=15, max_depth=4, min_samples_split=5, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=15, learning_rate=0.1, max_depth=3, random_state=42)
}

# Evaluate metrics dictionary
comparison_report = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluation
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    comparison_report[name] = {
        "R2_Score": r2,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "predictions": y_pred
    }
    
    print(f"=== {name} ===")
    print(f"R2 Score (Variance Explained): {r2:.5f}")
    print(f"MAE: {mae:.5f}")
    print(f"MSE: {mse:.5f}")
    print(f"RMSE: {rmse:.5f}\n")

In [ ]:
# Compare and select the best model based on maximum R2 Score
best_model_name = max(comparison_report, key=lambda k: comparison_report[k]["R2_Score"])
best_model = models[best_model_name]
best_predictions = comparison_report[best_model_name]["predictions"]

print(f"*** Best Performing Model: {best_model_name} ***")

## 6. Model Visual Diagnostics (Actual vs Predicted, Residuals, Importance)

In [ ]:
# 6.1 Actual vs Predicted Plot
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=best_predictions, color="indigo", alpha=0.9)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.title(f"Actual vs Predicted HDI ({best_model_name})", fontsize=13, fontweight="bold")
plt.xlabel("Actual HDI Values")
plt.ylabel("Predicted HDI Values")
plt.savefig("plots/actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 6.2 Residual Plot
residuals = y_test - best_predictions
plt.figure(figsize=(10, 5))
sns.scatterplot(x=best_predictions, y=residuals, color="teal", alpha=0.8)
plt.axhline(y=0, color="red", linestyle="--", lw=1.5)
plt.title(f"Model Prediction Residuals ({best_model_name})", fontsize=13, fontweight="bold")
plt.xlabel("Predicted HDI values")
plt.ylabel("Residual Error (Actual - Predicted)")
plt.savefig("plots/residuals_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 6.3 Feature Importance Chart
if hasattr(best_model, "feature_importances_"):
    importance = best_model.feature_importances_
    imp_df = pd.DataFrame({"Feature": features, "Importance": importance}).sort_values(by="Importance", ascending=False)
    
    plt.figure(figsize=(8, 4))
    sns.barplot(x="Importance", y="Feature", data=imp_df, palette="viridis")
    plt.title(f"Feature Importance Profiles: {best_model_name}", fontsize=13, fontweight="bold")
    plt.xlabel("Relative Importance Magnitude")
    plt.ylabel("Socioeconomic Indicator")
    plt.savefig("plots/feature_importances.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print(f"Linear Regression does not feature a tree split importance array. Coefficients are: {best_model.coef_}")

## 7. Module 6: Model Serialization (Pickling)

In [ ]:
# Create model persistence folder
os.makedirs("../model", exist_ok=True)
os.makedirs("model", exist_ok=True)

# Save trained best model and scaler using pickle
with open("model/model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and Scaler successfully serialized using pickle!")